# Toy Model: Ecology of Conspiracy Contagions

This notebook implements the minimal formal model from the paper. Three equations
operate in a shared belief space:

1. **Adoption hazard** — likelihood of adoption depends on distance in belief space
2. **Content-specific reshaping** — adoption shifts the belief system toward the narrative's signature
3. **Homeostatic recovery** — a linear spring pulls the belief system back toward baseline

We simulate **500 independent users** with no social network. All analysis is
derived from the simulation output — the adoption order, timing, and belief
states all emerge from the stochastic process.

## Step 1: Setup

We work in a **2-dimensional belief space**. Each conspiracy narrative has a
fixed position (its *signature*), and each user has a moving position (their
*belief state*).

- **Cluster A**: narratives A1, A2, A3 — grouped in the upper-left
- **Cluster B**: narratives B1, B2, B3 — grouped in the upper-right
- **Baseline θ₀**: the origin — where all users start

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import os
from tqdm import tqdm

np.random.seed(42)
os.makedirs('../../figures/formal_models', exist_ok=True)

# ── Narrative signatures: fixed positions in 2D belief space ──
#
# These are the "addresses" of each conspiracy in belief space.
# Narratives within a cluster are close together; clusters are far apart.

signatures = {
    'A1': np.array([0.0,  1.0]),
    'A2': np.array([0.15, 1.15]),
    'A3': np.array([-0.1, 1.2]),
    'B1': np.array([1.3,  1.0]),
    'B2': np.array([1.45, 1.15]),
    'B3': np.array([1.2,  1.2]),
}

# Cluster membership
clusters = {'A1': 0, 'A2': 0, 'A3': 0, 'B1': 1, 'B2': 1, 'B3': 1}
names = list(signatures.keys())
n_narratives = len(names)

# Baseline: starting position for all users
baseline = np.array([0.0, 0.0])

# Print summary
print("Narrative positions:")
for name in names:
    d = np.linalg.norm(signatures[name] - baseline)
    cl = 'A' if clusters[name] == 0 else 'B'
    print(f"  {name}  cluster {cl}  position {signatures[name]}  "
          f"distance from baseline = {d:.2f}")

In [ ]:
from pathlib import Path

INTERMEDIATE_DIR = Path("../../intermediate_files")
INTERMEDIATE_DIR.mkdir(exist_ok=True)

def intermediate_path(filename):
    return INTERMEDIATE_DIR / filename


## Step 2: Model Parameters

| Parameter | Symbol | Value | Role |
|-----------|--------|-------|------|
| Base adoption rate | μ | 0.05 | Scales overall adoption speed |
| Resistance scale | ρ | 1.0 | How quickly hazard drops with distance |
| Reshaping strength | α | 0.4 | How far belief shifts toward adopted narrative |
| Spring constant | k | 0.01 | How fast belief reverts toward baseline |

In [ ]:
# ── Model parameters ──
mu = 0.05        # base adoption rate
rho = 0.8        # resistance scale (Gaussian kernel bandwidth)
alpha = 0      # reshaping strength (0 = no shift, 1 = full shift)
k_spring = 0.01  # linear spring constant for recovery

# ── Simulation settings ──
n_users = 500    # independent users (no network)
t_max = 1000      # maximum time (hours)
dt = 1.0         # time step (hours)

## Step 3: The Three Equations

### Equation 1: Adoption hazard

$$h = \mu \cdot \exp\!\left(-\frac{\|\sigma - \theta\|^2}{\rho}\right)$$

Close in belief space → high hazard → easy to adopt.
Far in belief space → low hazard → hard to adopt.

In [ ]:
def adoption_hazard(theta, sigma):
    """
    Equation 1: Adoption hazard.
    
    The instantaneous rate at which a user at belief state theta
    adopts a narrative with signature sigma. Decays as a Gaussian
    with distance in belief space.
    """
    distance_sq = np.sum((sigma - theta) ** 2)
    return mu * np.exp(-distance_sq / rho)

### Equation 2: Content-specific reshaping

$$\theta^+ = (1 - \alpha)\,\theta^- + \alpha\,\sigma_c$$

Adoption pulls the belief state a fraction α toward the narrative's signature.
The shift is along exactly the dimensions the narrative engages.

In [ ]:
def reshape_belief(theta, sigma):
    """
    Equation 2: Content-specific reshaping.
    
    After adopting a narrative with signature sigma, the belief state
    moves a fraction alpha toward that signature. Returns the new
    belief state.
    """
    return (1 - alpha) * theta + alpha * sigma

### Equation 3: Homeostatic recovery (linear spring)

$$\frac{d\theta}{dt} = k\,(\theta_0 - \theta)$$

Between adoptions, a linear restoring force pulls the belief state back
toward baseline. Produces exponential decay: $\theta(t) = \theta_0 + \Delta\theta \cdot e^{-kt}$.

In [ ]:
def recovery_step(theta, theta_0, dt_step): #note: discrete approximation
    """
    Equation 3: Homeostatic recovery (linear spring).
    
    Pulls the belief state toward baseline by a small increment.
    Applied at every time step between adoption events.
    Returns the updated belief state.
    """
    return theta + k_spring * (theta_0 - theta) * dt_step

## Step 4: Run the Simulation

Each user evolves independently — no social network, no prescribed adoption order.
At each time step:

1. **Spring recovery** pulls belief toward baseline
2. **Hazards** computed for each unadopted narrative
3. **Stochastic adoption**: with probability from total hazard, one narrative is chosen
4. **Reshaping**: belief shifts toward the adopted narrative's signature

We record two things:

- **`records`**: the adoption history per user (for settler analysis and first-adoption counts)
- **`adoption_events`**: a flat list of every adoption event across all users, storing the
  post-adoption belief state and which narratives remain unadopted (for hazard ratio analysis)

In [ ]:
# Pre-compute signature array for vectorised hazard computation
sig_array = np.array([signatures[name] for name in names])

# ── Storage ──
#
# records: one list per user, each entry is (narrative_name, time)
#   -> used for settler analysis, first-adoption counts, transition times
#
# adoption_events: one entry per adoption event across ALL users, storing:
#   - depth: is this the user's 1st, 2nd, 3rd... adoption?
#   - theta_post: belief state immediately after reshaping
#   - remaining: list of narrative names not yet adopted by this user
#   - name: which narrative was adopted
#   -> used for hazard ratio decay curves and potency analysis

records = []
adoption_events = []

print(f"Simulating {n_users} independent users...")

for u in range(n_users):
    # Each user starts near baseline with tiny random noise
    theta = baseline.copy() #+ np.random.normal(0, 0.02, 2)
    theta_0 = theta.copy()  # this user's personal baseline
    
    adopted_names = []  # which narratives this user has adopted (in order)
    already_adopted = np.zeros(n_narratives, dtype=bool)
    t = 0.0
    
    while t < t_max and not already_adopted.all():
        # ── Step A: Compute hazards for all unadopted narratives ──
        distances_sq = np.sum((sig_array - theta) ** 2, axis=1)
        hazards = mu * np.exp(-distances_sq / rho)
        hazards[already_adopted] = 0.0
        total_hazard = hazards.sum()
        
        # ── Step B: Stochastic adoption ──
        if total_hazard > 0:
            prob_adopt = 1 - np.exp(-total_hazard * dt)
            
            if np.random.random() < prob_adopt:
                # Choose which narrative (proportional to hazards)
                probs = hazards / total_hazard
                chosen_idx = np.random.choice(n_narratives, p=probs)
                chosen_name = names[chosen_idx]
                
                # Record the adoption in the user's history
                adopted_names.append(chosen_name)
                already_adopted[chosen_idx] = True
                
                # ── Reshape belief toward adopted narrative ──
                theta = (1 - alpha) * theta + alpha * sig_array[chosen_idx]
                
                # ── Record this event for analysis ──
                remaining = [names[j] for j in range(n_narratives)
                             if not already_adopted[j]]
                adoption_events.append({
                    'name': chosen_name,
                    'time': t,
                    'depth': len(adopted_names),  # 1st, 2nd, 3rd...
                    'theta_post': theta.copy(),
                    'remaining': remaining,
                    'theta_0': theta_0.copy(),
                })
        
        # ── Step C: Spring recovery ──
        theta = theta + k_spring * (theta_0 - theta) * dt
        
        t += dt
    
    # Store this user's adoption sequence
    records.append([(name, evt['time'])
                     for name, evt in zip(adopted_names,
                        [e for e in adoption_events[-len(adopted_names):]])])

print(f"Done. {len(adoption_events)} total adoption events across {n_users} users.")
print()

# Summary: how many narratives did users adopt?
for k in range(n_narratives + 1):
    n = sum(1 for r in records if len(r) >= k)
    print(f"  Users who adopted >= {k}: {n} ({100*n/n_users:.1f}%)")

## Step 5: Visualise the Belief Space

Plot the narrative positions and overlay actual adoption trajectories from
5 simulated users. Each arrow shows a reshaping event — the belief state
jumping toward the adopted narrative's signature. The order is whatever
the stochastic process produced.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Colours
cluster_colors = {0: '#4C72B0', 1: '#DD8452'}
pal = {'A1':'#2d4a8a', 'A2':'#4C72B0', 'A3':'#7a9fd4',
       'B1':'#b05a20', 'B2':'#DD8452', 'B3':'#e8a76c'}

# Unique colors for each user trajectory
user_colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']

# Plot narrative signatures
for name in names:
    col = cluster_colors[clusters[name]]
    ax.scatter(*signatures[name], s=200, c=col, edgecolors='black',
               linewidth=1, zorder=5)
    ax.annotate(name, signatures[name], textcoords="offset points",
                xytext=(10, 8), fontsize=12, fontweight='bold', color=col)

# Baseline
ax.scatter(*baseline, s=150, c='black', marker='*', zorder=5)
ax.annotate(r'Baseline $\theta_0$', baseline,
            textcoords="offset points", xytext=(10, -15), fontsize=11)

# Overlay trajectories from first 5 users
user_event_idx = 0
for u in range(min(5, len(records))):
    n_adopted = len(records[u])
    if n_adopted == 0:
        continue
    
    # Get this user's events
    user_events = adoption_events[user_event_idx:user_event_idx + n_adopted]
    user_event_idx += n_adopted
    
    # Build trajectory: baseline -> first theta_post -> second theta_post -> ...
    points = [baseline.copy()]
    for evt in user_events:
        points.append(evt['theta_post'].copy())
    
    # Build adoption order label
    order = ' -> '.join([r[0] for r in records[u]])
    ucol = user_colors[u % len(user_colors)]
    
    # Draw arrows with this user's unique color
    for i in range(len(points) - 1):
        ax.annotate('', xy=points[i+1], xytext=points[i],
                    arrowprops=dict(arrowstyle='->', color=ucol,
                                    lw=1.5, alpha=0.5))
    
    # Invisible line for legend entry
    ax.plot([], [], color=ucol, lw=1.5, alpha=0.7,
            label=f'User {u+1}: {order}')

# Print adoption orders for these users
print("Adoption orders (first 5 users):")
for u in range(min(5, len(records))):
    order = ' -> '.join([r[0] for r in records[u]])
    print(f"  User {u+1}: {order}")

ax.set_xlabel('Belief dimension 1', fontsize=12)
ax.set_ylabel('Belief dimension 2', fontsize=12)
ax.set_title('Simulated adoption trajectories in belief space', fontsize=13)
ax.set_aspect('equal')
ax.legend(fontsize=8, loc='lower right', framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.5, 1.8)
ax.set_ylim(-0.3, 1.5)
plt.tight_layout()
plt.savefig('../../figures/formal_models/01_belief_space.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../../figures/formal_models/01_belief_space.png")

## Step 6: Hazard Ratio Decay Curves (P1, P3)

For each adoption event in the simulation, we have the **actual post-adoption
belief state** and the **actual set of remaining narratives**. We use these to
compute hazard ratio decay curves:

1. Take the recorded `theta_post` from the adoption event
2. Run the spring forward **deterministically** for 400 hours (no further adoptions)
3. At each time point, compute the hazard for each remaining narrative from
   the recovering belief state, and divide by the hazard from baseline
4. Average across the remaining set to get the mean HR at that time point

This gives one decay curve per adoption event. We then **average across all
events at the same depth** (all 1st adoptions, all 2nd adoptions, etc.) to
get one mean curve per depth.

The result shows:
- **P1**: every curve starts above 1.0 → adoption facilitates further adoption
- **P3**: peaks increase with depth → cumulative amplification

In [ ]:
# ── Compute hazard ratio decay curves by adoption depth ──
#
# For each adoption event, we deterministically evolve the recorded
# theta_post via the spring (no further stochastic adoptions), and
# track how the hazard ratio for remaining narratives decays over time.

# Time grid for the decay curves (0 to 400 hours, 0.5-hour steps)
decay_dt = dt
decay_times = np.arange(0, t_max, decay_dt)
n_timepoints = len(decay_times)

# Group events by depth
max_depth = min(5, n_narratives - 1)  # max 5 curves (6 narratives - 1)
events_by_depth = defaultdict(list)
for evt in adoption_events:
    if evt['depth'] <= max_depth:
        events_by_depth[evt['depth']].append(evt)

# For each depth, compute the average decay curve
decay_curves = {}  # depth -> array of mean HRs at each time point

for depth in tqdm(sorted(events_by_depth.keys())):
    events = events_by_depth[depth]
    
    # Collect one decay curve per event
    all_curves = []
    
    for evt in events:
        theta_post = evt['theta_post']
        remaining = evt['remaining']
        user_theta_0 = evt['theta_0']
        
        if len(remaining) == 0:
            continue
        
        # Baseline reference: mean hazard from this user's theta_0 to remaining narratives
        baseline_hazards = [adoption_hazard(user_theta_0, signatures[r])
                           for r in remaining]
        baseline_mean = np.mean(baseline_hazards)
        
        if baseline_mean == 0:
            continue
        
        # Run the spring forward from theta_post
        theta_t = theta_post.copy()
        curve = np.zeros(n_timepoints)
        
        for i in range(n_timepoints):
            # Compute mean hazard from current theta_t to remaining narratives
            current_hazards = [adoption_hazard(theta_t, signatures[r])
                              for r in remaining]
            curve[i] = np.mean(current_hazards) / baseline_mean
            
            # Spring recovery step (pull toward this user's personal baseline)
            theta_t = theta_t + k_spring * (user_theta_0 - theta_t) * decay_dt
        
        all_curves.append(curve)
    
    # Average across all events at this depth
    if all_curves:
        decay_curves[depth] = np.mean(all_curves, axis=0)

# Print peak hazard ratios
print("Peak hazard ratio (mean across users, immediately after adoption):")
for depth in sorted(decay_curves.keys()):
    peak = decay_curves[depth][0]
    print(f"  After adoption {depth}: {peak:.2f}x baseline")

In [ ]:
# ── Plot the averaged decay curves ──

fig, ax = plt.subplots(figsize=(9, 5))

colors = ['#1b9e77', '#d95f02', '#7570b3', '#e7298a', '#66a61e']

for depth in sorted(decay_curves.keys()):
    curve = decay_curves[depth]
    n_events = len(events_by_depth[depth])
    suffix = {1:'st', 2:'nd', 3:'rd'}.get(depth, 'th')
    label = f'After {depth}{suffix} adoption (n={n_events})'
    ax.plot(decay_times, curve, color=colors[(depth-1) % len(colors)],
            linewidth=2, label=label)

ax.axhline(1.0, color='grey', linestyle='--', linewidth=1,
           label='Baseline (no adoption)')
ax.set_xlabel('Hours since adoption', fontsize=12)
ax.set_ylabel('Mean hazard ratio vs. never-adopted baseline', fontsize=12)
ax.set_title('Hazard Ratio Decay by Adoption Depth (P1, P3)', fontsize=13)
ax.set_xlim(0, 300)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../../figures/formal_models/02_hazard_decay.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../../figures/formal_models/02_hazard_decay.png")

## Step 7: Interaction Windows (P4)

The **interaction window** is the time it takes for the hazard ratio to decay
back below a threshold (1.1× baseline). We read this directly from the
averaged decay curves computed above.

If windows expand with depth, it means the belief system stays displaced
longer after more cumulative reshaping.

In [ ]:
# ── Extract window duration from each averaged decay curve ──

threshold = 1.1
windows = {}

for depth in sorted(decay_curves.keys()):
    curve = decay_curves[depth]
    # Find first time point where curve drops below threshold
    below = np.where(curve < threshold)[0]
    if len(below) > 0:
        windows[depth] = decay_times[below[0]]
    else:
        windows[depth] = decay_times[-1]  # never drops below

print(f"Interaction window duration (hours to decay below {threshold}x baseline):")
for depth in sorted(windows.keys()):
    print(f"  After adoption {depth}: {windows[depth]:.0f} hours")

# ── Plot ──
fig, ax = plt.subplots(figsize=(6, 4))

depths = sorted(windows.keys())
vals = [windows[d] for d in depths]
ax.bar(depths, vals,
       color=[colors[(d-1) % len(colors)] for d in depths],
       edgecolor='black', linewidth=0.8, width=0.6)

for d, v in zip(depths, vals):
    ax.text(d, v + max(vals)*0.02, f'{v:.0f}h',
            ha='center', fontsize=10, fontweight='bold')

suffix_map = {1:'st', 2:'nd', 3:'rd'}
ax.set_xlabel('Adoption number', fontsize=12)
ax.set_ylabel('Window duration (hours)', fontsize=12)
ax.set_title('Expanding Interaction Windows (P4)', fontsize=13)
ax.set_xticks(depths)
ax.set_xticklabels([f'{d}{suffix_map.get(d,"th")}' for d in depths])
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../../figures/formal_models/03_interaction_windows.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../../figures/formal_models/03_interaction_windows.png")

## Step 8: Contagiousness–Potency Tradeoff

We measure two properties of each narrative from the simulation:

**Contagiousness**: how often a narrative is adopted **first** (count of users
whose first adoption was this narrative). Narratives close to baseline have
higher hazard from the starting position, so they get adopted first more often.

**Potency**: when a user adopts narrative c, how much does the hazard for
all remaining narratives increase? We compute the mean hazard ratio across
remaining narratives at the moment of adoption, then average across all users
who adopted c. Higher potency = adopting this narrative boosts susceptibility
to everything else more strongly.

Both are expressed **relative to A1** (reference at 1.0), mirroring the empirical
Figure where Fake News is the reference.

In [ ]:
# ── Contagiousness: first-adoption count per narrative ──

first_counts = defaultdict(int)
for r in records:
    if len(r) > 0:
        first_counts[r[0][0]] += 1

# ── Potency: mean HR boost at the moment of adoption ──
#
# For each FIRST adoption event where narrative c was adopted, compute:
#   mean over remaining narratives r of: hazard(theta_post, sigma_r) / hazard(theta_0, sigma_r)
# Then average this across all first-adoption events where c was adopted.

potency_values = defaultdict(list)  # name -> list of mean HR values

for evt in adoption_events:
    if evt['depth'] != 1:
        continue
    
    remaining = evt['remaining']
    if len(remaining) == 0:
        continue
    
    theta_post = evt['theta_post']
    user_theta_0 = evt['theta_0']
    
    # HR for each remaining narrative
    hrs = []
    for r in remaining:
        h_post = adoption_hazard(theta_post, signatures[r])
        h_base = adoption_hazard(user_theta_0, signatures[r])
        if h_base > 0:
            hrs.append(h_post / h_base)
    
    if hrs:
        potency_values[evt['name']].append(np.mean(hrs))

# Compute mean potency per narrative
potency = {}
for name in names:
    if potency_values[name]:
        potency[name] = np.mean(potency_values[name])
    else:
        potency[name] = 0.0

# ── Express relative to A1 ──
ref = 'A1'
ref_cong = first_counts[ref]
ref_pot = potency[ref]

print(f"Reference: {ref} (contagiousness = {ref_cong} users, "
      f"potency = {ref_pot:.2f}x)")
print()
print(f"{'Name':<6} {'1st adopt':<10} {'Rel. cong.':<12} "
      f"{'Mean HR':<10} {'Rel. pot.':<10}")
print("-" * 50)
for name in names:
    rel_c = first_counts[name] / ref_cong if ref_cong else 0
    rel_p = potency[name] / ref_pot if ref_pot else 0
    print(f"  {name:<4} {first_counts[name]:<10} {rel_c:<12.3f} "
          f"{potency[name]:<10.2f} {rel_p:<10.3f}")

In [ ]:
# ── Plot the tradeoff scatter ──

fig, ax = plt.subplots(figsize=(7, 5))

for name in names:
    rel_c = first_counts.get(name, 0) / ref_cong if ref_cong else 0
    rel_p = potency[name] / ref_pot if ref_pot else 0
    col = cluster_colors[clusters[name]]
    ax.scatter(rel_c, rel_p, s=200, c=col, edgecolors='black',
               linewidth=1, zorder=5)
    ax.annotate(name, (rel_c, rel_p), textcoords="offset points",
                xytext=(8, 8), fontsize=12, fontweight='bold', color=col)

# Reference lines at (1, 1)
ax.axhline(1.0, color='grey', linestyle=':', linewidth=0.8, alpha=0.5)
ax.axvline(1.0, color='grey', linestyle=':', linewidth=0.8, alpha=0.5)
ax.annotate(f'reference ({ref})', (1.0, 1.0), textcoords="offset points",
            xytext=(-70, -15), fontsize=9, color='grey', style='italic')

ax.set_xlabel(f'Contagiousness (first-adoption rate, relative to {ref})',
              fontsize=11)
ax.set_ylabel(f'Potency (mean HR boost, relative to {ref})', fontsize=11)
ax.set_title('Contagiousness–Potency Tradeoff (from simulation)', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../../figures/formal_models/04_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../../figures/formal_models/04_tradeoff.png")

## Step 9: Settler Dynamics (P2, P5)

We use a **sliding 4-element window** (matching the empirical `compute_settler_effect()`)
to detect the AABB pattern: two consecutive adoptions in cluster A, then two in cluster B.
Only windows where cluster B has not been previously visited are counted.

From each matched window we extract three **paired** transition times:

- **Pre-jump** (old → old): time between positions i−1 and i (both in cluster A)
- **The jump** (old → new): time between positions i and i+1 (A → B)
- **Post-jump** (new → new): time between positions i+1 and i+2 (both in cluster B)

If the theory is correct: pre-jump is fast (belief already reshaped in that region),
the jump is slow (crossing the cluster boundary), and post-jump is fast again
(one adoption has started reshaping in the new region — the settler effect).

In [ ]:
# ── Classify transitions using AABB sliding window ──
# Matches the empirical compute_settler_effect() pattern:
# Position i-1, i in cluster A (old, old)
# Position i+1, i+2 in cluster B (new, new)
# Extracts three paired transition times per window.

pre_jump_times = []
the_jump_times = []
post_jump_times = []

for user_record in records:
    if len(user_record) < 4:
        continue

    seen_clusters = {clusters[user_record[0][0]]}

    for i in range(1, len(user_record) - 2):
        seen_clusters.add(clusters[user_record[i][0]])

        # Get clusters for the 4-element window [i-1, i, i+1, i+2]
        cl = [clusters[user_record[j][0]] for j in range(i - 1, i + 3)]

        is_stable_before = cl[0] == cl[1]   # old, old
        is_switching = cl[1] != cl[2]        # old != new
        is_stable_after = cl[2] == cl[3]     # new, new

        if is_stable_before and is_switching and is_stable_after:
            # Only count if new cluster hasn't been visited before
            if cl[2] in seen_clusters:
                continue
            pre_jump_times.append(user_record[i][1] - user_record[i-1][1])
            the_jump_times.append(user_record[i+1][1] - user_record[i][1])
            post_jump_times.append(user_record[i+2][1] - user_record[i+1][1])

print("Settler dynamics — AABB pattern (median transition times):")
print(f"  Pre-jump (within old):   {np.median(pre_jump_times):5.0f} hours  "
      f"(n={len(pre_jump_times)})")
print(f"  The jump (cross-cluster): {np.median(the_jump_times):5.0f} hours  "
      f"(n={len(the_jump_times)})")
print(f"  Post-jump (settling new): {np.median(post_jump_times):5.0f} hours  "
      f"(n={len(post_jump_times)})")

In [ ]:
# ── Plot settler dynamics ──

fig, ax = plt.subplots(figsize=(6, 4))

categories = ['Within old\ncluster', 'Jump to\nnew cluster',
              'Settling in\nnew cluster']
medians = [np.median(pre_jump_times), np.median(the_jump_times),
           np.median(post_jump_times)]
counts = [len(pre_jump_times), len(the_jump_times), len(post_jump_times)]
bar_colors = ['#4C72B0', '#999999', '#DD8452']

bars = ax.bar(range(3), medians, color=bar_colors,
              edgecolor='black', linewidth=0.8, width=0.6)
ax.set_xticks(range(3))
ax.set_xticklabels(categories, fontsize=10)
ax.set_ylabel('Median transition time (hours)', fontsize=12)
ax.set_title('Settler Dynamics (P2, P5)', fontsize=13)
ax.set_ylim(0, max(medians)*1.4)

for i, (bar, n) in enumerate(zip(bars, counts)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{medians[i]:.0f}h\n(n={n})', ha='center', fontsize=9,
            color='#555')

ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../../figures/formal_models/05_settler_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../../figures/formal_models/05_settler_dynamics.png")

## Step 9b: Temporal Hazard Shift (cf. Empirical Figure 5)

Snapshot the hazard ratio at **1h, 24h, and 72h** after adoption for each depth.
This mirrors the empirical Figure 5 (grouped bar chart comparing Weibull baselines
across sequential models). Here the "models" are adoption depths 1–5, and the
hazard ratios come from the deterministic decay curves computed in Step 6.

In [ ]:
# ── Temporal Hazard Shift: grouped bar chart ──

eval_times = [1.0, 24.0, 72.0]
eval_labels = [f'{int(t)}h' for t in eval_times]

# Extract HR at each eval time from the decay curves
depth_colors = {
    1: '#E24A33', 2: '#348ABD', 3: '#988ED5', 4: '#8EBA42', 5: '#FFB000',
}
depth_labels = {
    1: 'After 1st adoption', 2: 'After 2nd adoption', 3: 'After 3rd adoption',
    4: 'After 4th adoption', 5: 'After 5th adoption',
}

fig, ax = plt.subplots(figsize=(10, 5))

depths_available = sorted(decay_curves.keys())
n_depths = len(depths_available)
x = np.arange(len(eval_times))
width = 0.8 / n_depths

for i, depth in enumerate(depths_available):
    curve = decay_curves[depth]
    ratios = []
    for t in eval_times:
        idx = int(t / decay_dt)
        idx = min(idx, len(curve) - 1)
        ratios.append(curve[idx])
    
    offset = (i - (n_depths - 1) / 2) * width
    color = depth_colors.get(depth, f'C{i}')
    label = depth_labels.get(depth, f'After adoption {depth}')
    bars = ax.bar(x + offset, ratios, width, color=color,
                  edgecolor='black', alpha=0.85, label=label)
    
    for bar, ratio in zip(bars, ratios):
        ax.annotate(f'{ratio:.1f}x',
                    xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 4), textcoords='offset points',
                    ha='center', fontsize=7, fontweight='bold')

ax.axhline(1.0, color='grey', linestyle='--', linewidth=1.2,
           label='Baseline (no adoption)')
ax.set_xticks(x)
ax.set_xticklabels(eval_labels)
ax.set_xlabel('Time since adoption (hours)', fontsize=12)
ax.set_ylabel('Hazard ratio vs. never-adopted baseline', fontsize=12)
ax.set_title('Temporal Hazard Shift by Adoption Depth', fontsize=13)
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../../figures/formal_models/07_temporal_hazard_shift.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../../figures/formal_models/07_temporal_hazard_shift.png")

## Summary

Three equations, one shared belief space, **no social network**, **no prescribed
adoption order**. The stochastic process decides who adopts what and when.
All results above are derived from the simulation output:

| Pattern | What we see | How we measure it |
|---------|-------------|-------------------|
| **P1** Facilitation | Hazard curves above 1.0 after every adoption | Mean HR decay curves by depth |
| **P2** Semantic proximity | Within-cluster faster than cross-cluster | Settler transition times + cognitive barrier |
| **P3** Amplification | Peaks increase with depth | Peak HR from averaged curves + temporal hazard shift |
| **P4** Expanding windows | Windows grow with depth | Time to decay below 1.1× baseline |
| **P5** Settler dynamics | Jump slow, post-settler fast | Transition classification |

All figures saved to the `figures/` directory.

In [ ]:
# ── Cognitive Barrier: within-cluster vs between-cluster ──
# Independent computation matching empirical compute_semantic_barrier_analysis().
# Examines every consecutive pair of adoptions; between-cluster only counts
# first visits to new clusters.

within_cluster_times = []
between_cluster_times = []

for user_record in records:
    if len(user_record) < 2:
        continue

    seen_clusters = {clusters[user_record[0][0]]}

    for i in range(len(user_record) - 1):
        curr_cluster = clusters[user_record[i][0]]
        next_cluster = clusters[user_record[i + 1][0]]
        gap = user_record[i + 1][1] - user_record[i][1]

        if curr_cluster == next_cluster:
            within_cluster_times.append(gap)
        elif next_cluster not in seen_clusters:
            between_cluster_times.append(gap)

        seen_clusters.add(next_cluster)

# Bootstrap 95% CI on the median
def bootstrap_median_ci(data, n_boot=2000, ci=0.95):
    data = np.array(data)
    medians = np.array([np.median(np.random.choice(data, size=len(data), replace=True))
                        for _ in range(n_boot)])
    med = np.median(data)
    alpha = (1 - ci) / 2
    lo, hi = np.percentile(medians, [100 * alpha, 100 * (1 - alpha)])
    return med, med - lo, hi - med

labels = ['Within Cluster', 'Between Clusters']
bar_colors = ['#4A90E2', '#D0021B']

medians, err_lo, err_hi = [], [], []
for data in [within_cluster_times, between_cluster_times]:
    med, el, eh = bootstrap_median_ci(data, n_boot=2000)
    medians.append(med)
    err_lo.append(el)
    err_hi.append(eh)

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(labels, medians, yerr=[err_lo, err_hi], capsize=10,
       color=bar_colors, edgecolor='black', alpha=0.8)
ax.set_ylabel('Median hours to adoption', fontsize=12)
ax.set_title('Jumping to a New Cluster is Slower\n(Median with 95% Bootstrap CI)',
             fontsize=13)

for i, v in enumerate(medians):
    ax.text(i, v, f'{v:.1f}h', ha='center', va='bottom',
            fontweight='bold', fontsize=11)

n_within = len(within_cluster_times)
n_between = len(between_cluster_times)
ax.text(0, 0.02, f'n={n_within}', ha='center', va='bottom',
        transform=ax.get_xaxis_transform(), fontsize=9, color='#555')
ax.text(1, 0.02, f'n={n_between}', ha='center', va='bottom',
        transform=ax.get_xaxis_transform(), fontsize=9, color='#555')

ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../../figures/formal_models/08_cognitive_barrier.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Within Cluster: median={medians[0]:.1f}h (n={n_within})")
print(f"Between Clusters: median={medians[1]:.1f}h (n={n_between})")
print("Saved: ../../figures/formal_models/08_cognitive_barrier.png")

## Step 10: First Adoption Frequency

Which narrative gets adopted first? The model predicts that narratives closer
to baseline are adopted first more often, because their hazard from the
starting position is higher.

In [ ]:
sorted_names = sorted(first_counts.keys(),
                       key=lambda c: -first_counts[c])

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(range(len(sorted_names)),
        [first_counts[c] for c in sorted_names],
        color=[cluster_colors[clusters[c]] for c in sorted_names],
        edgecolor='black', linewidth=0.8, height=0.55)
ax.set_yticks(range(len(sorted_names)))
ax.set_yticklabels(
    [f'{c}  (d={np.linalg.norm(signatures[c]-baseline):.2f})'
     for c in sorted_names], fontsize=11)
ax.set_xlabel('Number of users (first adoption)', fontsize=12)
ax.set_title('First Adoption by Narrative', fontsize=13)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('../../figures/formal_models/06_first_adoption.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../../figures/formal_models/06_first_adoption.png")

## Summary

Three equations, one shared belief space, **no social network**, **no prescribed
adoption order**. The stochastic process decides who adopts what and when.
All results above are derived from the simulation output:

| Pattern | What we see | How we measure it |
|---------|-------------|-------------------|
| **P1** Facilitation | Hazard curves above 1.0 after every adoption | Mean HR decay curves by depth |
| **P2** Semantic proximity | Within-cluster faster than cross-cluster | Settler transition times |
| **P3** Amplification | Peaks increase with depth | Peak HR from averaged curves |
| **P4** Expanding windows | Windows grow with depth | Time to decay below 1.1× baseline |
| **P5** Settler dynamics | Jump slow, post-settler fast | Transition classification |

All figures saved to the `figures/` directory.

## Combined Figure

All key results in a single 3x2 panel for the paper.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 18))
panel_labels = ['a', 'b', 'c', 'd', 'e', 'f']

# ── (a) Belief space trajectories ──────────────────────────────────────
ax = axes[0, 0]
cluster_colors_local = {0: '#4C72B0', 1: '#DD8452'}
user_colors_local = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']

for name in names:
    col = cluster_colors_local[clusters[name]]
    ax.scatter(*signatures[name], s=200, c=col, edgecolors='black', linewidth=1, zorder=5)
    ax.annotate(name, signatures[name], textcoords="offset points",
                xytext=(10, 8), fontsize=14, fontweight='bold', color=col)

ax.scatter(*baseline, s=150, c='black', marker='*', zorder=5)
ax.annotate(r'Baseline $\theta_0$', baseline,
            textcoords="offset points", xytext=(10, -15), fontsize=13)

user_event_idx = 0
for u in range(min(5, len(records))):
    n_adopted = len(records[u])
    if n_adopted == 0:
        continue
    user_events = adoption_events[user_event_idx:user_event_idx + n_adopted]
    user_event_idx += n_adopted
    points = [baseline.copy()]
    for evt in user_events:
        points.append(evt['theta_post'].copy())
    order = ' -> '.join([r[0] for r in records[u]])
    ucol = user_colors_local[u % len(user_colors_local)]
    for i in range(len(points) - 1):
        ax.annotate('', xy=points[i+1], xytext=points[i],
                    arrowprops=dict(arrowstyle='->', color=ucol, lw=1.5, alpha=0.5))
    ax.plot([], [], color=ucol, lw=1.5, alpha=0.7, label=f'User {u+1}: {order}')

ax.set_xlabel('Belief dimension 1', fontsize=14)
ax.set_ylabel('Belief dimension 2', fontsize=14)
ax.set_title('Adoption Trajectories in Belief Space', fontsize=15)
ax.set_aspect('equal')
ax.legend(fontsize=9, loc='lower right', framealpha=0.9)
ax.tick_params(labelsize=12)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.5, 1.8)
ax.set_ylim(-0.3, 1.5)

# ── (b) Hazard ratio decay curves ─────────────────────────────────────
ax = axes[0, 1]
decay_colors = ['#1b9e77', '#d95f02', '#7570b3', '#e7298a', '#66a61e']

for depth in sorted(decay_curves.keys()):
    curve = decay_curves[depth]
    n_events = len(events_by_depth[depth])
    suffix = {1:'st', 2:'nd', 3:'rd'}.get(depth, 'th')
    label = f'After {depth}{suffix} adoption (n={n_events})'
    ax.plot(decay_times, curve, color=decay_colors[(depth-1) % len(decay_colors)],
            linewidth=2, label=label)

ax.axhline(1.0, color='grey', linestyle='--', linewidth=1, label='Baseline')
ax.set_xlabel('Hours since adoption', fontsize=14)
ax.set_ylabel('Mean hazard ratio vs. baseline', fontsize=14)
ax.set_title('Hazard Ratio Decay by Adoption Depth', fontsize=15)
ax.set_xlim(0, 300)
ax.legend(fontsize=10)
ax.tick_params(labelsize=12)
ax.grid(True, alpha=0.3)

# ── (c) Interaction windows ───────────────────────────────────────────
ax = axes[1, 0]
depths_sorted = sorted(windows.keys())
vals = [windows[d] for d in depths_sorted]
ax.bar(depths_sorted, vals,
       color=[decay_colors[(d-1) % len(decay_colors)] for d in depths_sorted],
       edgecolor='black', linewidth=0.8, width=0.6)
for d, v in zip(depths_sorted, vals):
    ax.text(d, v + max(vals)*0.02, f'{v:.0f}h', ha='center', fontsize=12, fontweight='bold')
suffix_map = {1:'st', 2:'nd', 3:'rd'}
ax.set_xlabel('Adoption number', fontsize=14)
ax.set_ylabel('Window duration (hours)', fontsize=14)
ax.set_title('Expanding Interaction Windows', fontsize=15)
ax.set_xticks(depths_sorted)
ax.set_xticklabels([f'{d}{suffix_map.get(d,"th")}' for d in depths_sorted], fontsize=12)
ax.tick_params(labelsize=12)
ax.grid(True, alpha=0.3, axis='y')

# ── (d) Contagiousness–Potency tradeoff ───────────────────────────────
ax = axes[1, 1]
for name in names:
    rel_c = first_counts.get(name, 0) / ref_cong if ref_cong else 0
    rel_p = potency[name] / ref_pot if ref_pot else 0
    col = cluster_colors_local[clusters[name]]
    ax.scatter(rel_c, rel_p, s=200, c=col, edgecolors='black', linewidth=1, zorder=5)
    ax.annotate(name, (rel_c, rel_p), textcoords="offset points",
                xytext=(8, 8), fontsize=14, fontweight='bold', color=col)
ax.axhline(1.0, color='grey', linestyle=':', linewidth=0.8, alpha=0.5)
ax.axvline(1.0, color='grey', linestyle=':', linewidth=0.8, alpha=0.5)
ax.annotate(f'reference ({ref})', (1.0, 1.0), textcoords="offset points",
            xytext=(-70, -15), fontsize=11, color='grey', style='italic')
ax.set_xlabel(f'Contagiousness (rel. to {ref})', fontsize=14)
ax.set_ylabel(f'Potency (rel. to {ref})', fontsize=14)
ax.set_title('Contagiousness\u2013Potency Tradeoff', fontsize=15)
ax.tick_params(labelsize=12)
ax.grid(True, alpha=0.3)

# ── (e) Settler dynamics ──────────────────────────────────────────────
ax = axes[2, 0]
categories = ['Within old\ncluster', 'Jump to\nnew cluster', 'Settling in\nnew cluster']
settler_medians = [np.median(pre_jump_times), np.median(the_jump_times),
                   np.median(post_jump_times)]
settler_counts = [len(pre_jump_times), len(the_jump_times), len(post_jump_times)]
settler_colors = ['#4C72B0', '#999999', '#DD8452']
bars = ax.bar(range(3), settler_medians, color=settler_colors,
              edgecolor='black', linewidth=0.8, width=0.6)
ax.set_xticks(range(3))
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylabel('Median transition time (hours)', fontsize=14)
ax.set_title('Settler Dynamics', fontsize=15)
ax.set_ylim(0, max(settler_medians) * 1.4)
for i, (bar, n) in enumerate(zip(bars, settler_counts)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f'{settler_medians[i]:.0f}h\n(n={n})', ha='center', fontsize=11, color='#555')
ax.tick_params(labelsize=12)
ax.grid(True, alpha=0.3, axis='y')

# ── (f) Cognitive barrier (independent pairwise computation) ──────────
ax = axes[2, 1]

# Recompute independently from settler analysis
barrier_within = []
barrier_between = []
for user_record in records:
    if len(user_record) < 2:
        continue
    seen_cl = {clusters[user_record[0][0]]}
    for i in range(len(user_record) - 1):
        curr_cl = clusters[user_record[i][0]]
        next_cl = clusters[user_record[i + 1][0]]
        gap = user_record[i + 1][1] - user_record[i][1]
        if curr_cl == next_cl:
            barrier_within.append(gap)
        elif next_cl not in seen_cl:
            barrier_between.append(gap)
        seen_cl.add(next_cl)

barrier_labels = ['Within Cluster', 'Between Clusters']
barrier_colors = ['#4A90E2', '#D0021B']
barrier_medians, barrier_err_lo, barrier_err_hi = [], [], []
for data in [barrier_within, barrier_between]:
    med, el, eh = bootstrap_median_ci(data, n_boot=2000)
    barrier_medians.append(med)
    barrier_err_lo.append(el)
    barrier_err_hi.append(eh)

ax.bar(barrier_labels, barrier_medians,
       yerr=[barrier_err_lo, barrier_err_hi], capsize=10,
       color=barrier_colors, edgecolor='black', alpha=0.8)
ax.set_ylabel('Median hours to adoption', fontsize=14)
ax.set_title('Cognitive Barrier\n(Median with 95% Bootstrap CI)', fontsize=15)
for i, v in enumerate(barrier_medians):
    ax.text(i, v, f'{v:.1f}h', ha='center', va='bottom', fontweight='bold', fontsize=13)
ax.text(0, 0.02, f'n={len(barrier_within)}', ha='center', va='bottom',
        transform=ax.get_xaxis_transform(), fontsize=11, color='#555')
ax.text(1, 0.02, f'n={len(barrier_between)}', ha='center', va='bottom',
        transform=ax.get_xaxis_transform(), fontsize=11, color='#555')
ax.tick_params(labelsize=12)
ax.grid(True, alpha=0.3, axis='y')

# ── Add panel labels (a)–(f) ──────────────────────────────────────────
for idx, ax in enumerate(axes.flat):
    ax.text(-0.08, 1.08, f'({panel_labels[idx]})', transform=ax.transAxes,
            fontsize=18, fontweight='bold', va='top')

plt.tight_layout(h_pad=3.5, w_pad=3.0)
plt.savefig('../../figures/formal_models/combined_panel.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: ../../figures/formal_models/combined_panel.png")

In [ ]:
# ── Cache results for the main-text composite figure ─────────────────────
# Dumps the variables consumed by conspiracy_analysis.visualization.
# formal_model_panels so 03_main_text_figures.ipynb can render
# Figure 2 without re-running the simulation.
import joblib

results = {
    'names': names,
    'clusters': clusters,
    'signatures': signatures,
    'baseline': baseline,
    'records': records,
    'adoption_events': adoption_events,
    'decay_curves': decay_curves,
    'decay_times': decay_times,
    'events_by_depth': events_by_depth,
    'windows': windows,
    'first_counts': first_counts,
    'potency': potency,
    'ref': ref,
    'ref_cong': ref_cong,
    'ref_pot': ref_pot,
    'pre_jump_times': pre_jump_times,
    'the_jump_times': the_jump_times,
    'post_jump_times': post_jump_times,
}
joblib.dump(results, intermediate_path('no_interaction_model_results.pkl'))
print(f'Saved {len(results)} variables to no_interaction_model_results.pkl')